In [ ]:
import xarray as xr
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import numpy as np
from google.colab import files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
netcdf_path = "/content/drive/MyDrive/CONICET/Usos del suelo NLP/land2vec/data/landcover_timeseries_2000-2022.nc"
ds = xr.open_dataset(netcdf_path)

In [ ]:
ds.info()

xarray.Dataset {
dimensions:
	time = 23 ;
	lat = 12600 ;
	lon = 7920 ;

variables:
	float32 lccs_class(time, lat, lon) ;
	float32 lat(lat) ;
	float32 lon(lon) ;
	datetime64[ns] time(time) ;

// global attributes:
	:id = ESACCI-LC-L4-LCCS-Map-300m-P1Y-2000-v2.0.7cds ;
	:title = Land Cover Map of ESA CCI brokered by CDS ;
	:summary = This dataset characterizes the land cover of a particular year (see time_coverage). The land cover was derived from the analysis of satellite data time series of the full period. ;
	:type = ESACCI-LC-L4-LCCS-Map-300m-P1Y ;
	:project = Climate Change Initiative - European Space Agency ;
	:references = http://www.esa-landcover-cci.org/ ;
	:institution = UCLouvain ;
	:contact = https://www.ecmwf.int/en/about/contact-us/get-support ;
	:comment =  ;
	:Conventions = CF-1.6 ;
	:standard_name_vocabulary = NetCDF Climate and Forecast (CF) Standard Names version 21 ;
	:keywords = land cover classification,satellite,observation ;
	:keywords_vocabulary = NASA Global Cha

In [ ]:
seqs_frontier = pd.read_csv("/content/drive/MyDrive/CONICET/Usos del suelo NLP/land2vec/data/id_seqs_text_2000_2022_chaco_santiago_frontier.zip")
coords_frontier = pd.read_csv("/content/drive/MyDrive/CONICET/Usos del suelo NLP/land2vec/data/lat_long_df_chaco_santiago_frontier.zip")

In [ ]:
coords_frontier

,ID,latitude,longitude
0,0,-25.431944,-63.448612
1,1,-25.431944,-63.445835
2,2,-25.431944,-63.443054
3,3,-25.431944,-63.440277
4,4,-25.431944,-63.437500
...,...,...,...
1424452,1424452,-28.126389,-59.387500
1424453,1424453,-28.126389,-59.384724
1424454,1424454,-28.126389,-59.381943
1424455,1424455,-28.126389,-59.379166


In [ ]:
points = [[-63.44994621163554,-25.545424441292493],
[-63.34008293038554,-28.12819902009702],
[-59.37401847726054,-28.09428290164623],
[-59.63769035226054,-25.431378332142593]]

In [ ]:
# Create a GeoDataFrame from the points
points_gdf = gpd.GeoDataFrame(geometry=[Point(xy) for xy in points])

# Get the bounding box from the GeoDataFrame
minx, miny, maxx, maxy = points_gdf.total_bounds

# Crop the dataset using the bounding box
ds_cropped = ds.sel(lon=slice(minx, maxx), lat=slice(maxy, miny))

In [ ]:
def create_geo_info(ds):

  # Get latitude and longitude coordinates
  latitudes = ds['lat'].values
  longitudes = ds['lon'].values

  # Create a meshgrid for coordinates
  lon_grid, lat_grid = np.meshgrid(longitudes, latitudes)

  # Flatten the grids
  lon_flat = lon_grid.flatten()
  lat_flat = lat_grid.flatten()

  # Create a unique ID for each lat/lon combination
  num_pixels = len(lon_flat)
  pixel_ids = np.arange(num_pixels)

  return({'ID':pixel_ids,
          'num_pixels': num_pixels,
          'latitude_flat':lat_flat,
          'longitude_flat':lon_flat})


In [ ]:
def create_lat_long_df(ds):

  geo_info = create_geo_info(ds)

  # Create the lat_long dataframe
  lat_long_df = pd.DataFrame({
    'ID': geo_info['ID'],
    'latitude': geo_info['latitude_flat'],
    'longitude': geo_info['longitude_flat']
    })

  return(lat_long_df)

In [ ]:
def create_lccs_class_df(ds):

  geo_info = create_geo_info(ds)

  # Extract 'lccs_class' variable
  lccs_data = ds['lccs_class']

  # Flatten the lccs_class data for all time steps
  lccs_flat_all_times = lccs_data.values.reshape(-1, geo_info['num_pixels']).T

  # Create column names for lccs_class dataframe
  time_years = ds["time"].dt.year.values
  lccs_cols = [f'lccs_class_{year}' for year in time_years]

  # Create the lccs_class dataframe
  lccs_class_df = pd.DataFrame(lccs_flat_all_times, columns=lccs_cols)
  lccs_class_df['ID'] = geo_info['ID']

  return(lccs_class_df)

In [ ]:
lat_long_df = create_lat_long_df(ds)

In [ ]:
lccs_class_df = create_lccs_class_df(ds)

In [ ]:
lccs_class_df.head()

,lccs_class_2000,lccs_class_2001,lccs_class_2002,lccs_class_2003,lccs_class_2004,lccs_class_2005,lccs_class_2006,lccs_class_2007,lccs_class_2008,lccs_class_2009,...,lccs_class_2014,lccs_class_2015,lccs_class_2016,lccs_class_2017,lccs_class_2018,lccs_class_2019,lccs_class_2020,lccs_class_2021,lccs_class_2022,ID
0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,6.0,6.0,6.0,6.0,0
1,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,6.0,6.0,6.0,6.0,1
2,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,6.0,6.0,6.0,6.0,2
3,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,6.0,6.0,6.0,6.0,3
4,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,...,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,4


In [ ]:
%%time

# Mapping of numeric codes to letters (using integers as keys)
code_mapping = {
    1: 'A',
    2: 'F',
    3: 'G',
    4: 'Wt',
    5: 'U',
    6: 'Sh',
    7: 'Sp',
    8: 'B',
    9: 'Wa',
    0: 'Nd'
}

# Get all lccs_class columns
lccs_cols = [col for col in lccs_class_df.columns if 'lccs_class' in col]

# Process in chunks to avoid memory overflow
chunk_size = 100000  # Adjust if still crashing (try 50000)
result_list = []

for start_idx in range(0, len(lccs_class_df), chunk_size):
    end_idx = min(start_idx + chunk_size, len(lccs_class_df))
    chunk = lccs_class_df.iloc[start_idx:end_idx]

    # Apply mapping directly to integer values, then convert to string
    arrays = []
    for col in lccs_cols:
        # Map the integer values directly using the dict
        mapped_col = chunk[col].map(code_mapping).astype(str)
        arrays.append(mapped_col.values)

    # Concatenate with '-' using a simpler approach
    seqs = []
    for i in range(len(chunk)):
        seq = '-'.join([str(arrays[j][i]) for j in range(len(arrays))])
        seqs.append(seq)

    # Create chunk result
    chunk_result = pd.DataFrame({
        'ID': chunk['ID'].values,
        'seqs': seqs
    })

    result_list.append(chunk_result)

    print(f"Processed {end_idx} rows...")

Processed 100000 rows...
Processed 200000 rows...
Processed 300000 rows...
Processed 400000 rows...
Processed 500000 rows...
Processed 600000 rows...
Processed 700000 rows...
Processed 800000 rows...
Processed 900000 rows...
Processed 1000000 rows...
Processed 1100000 rows...
Processed 1200000 rows...
Processed 1300000 rows...
Processed 1400000 rows...
Processed 1424457 rows...
CPU times: user 5.3 s, sys: 34.3 ms, total: 5.34 s
Wall time: 5.33 s


In [ ]:
# Concatenate all chunks
result_df = pd.concat(result_list, ignore_index=True)

print(result_df.head())
print(f"Total rows: {len(result_df)}")

   ID                                               seqs
0   0  F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-Sh-Sh-Sh-Sh
1   1  F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-Sh-Sh-Sh-Sh
2   2  F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-Sh-Sh-Sh-Sh
3   3  F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-Sh-Sh-Sh-Sh
4   4  Sh-Sh-Sh-Sh-Sh-Sh-Sh-Sh-Sh-Sh-Sh-Sh-Sh-Sh-Sh-S...
Total rows: 1424457


In [ ]:
lat_long_df.to_csv('./lat_long_df_chaco_santiago_frontier.zip', index=False, compression='zip')
files.download('./lat_long_df_chaco_santiago_frontier.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
result_df.to_csv('./id_seqs_text_2000_2022_chaco_santiago_frontier.zip', index=False, compression='zip')
files.download('./id_seqs_text_2000_2022_chaco_santiago_frontier.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
result_df_filt = result_df[result_df["seqs"]!="Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa"]

In [ ]:
result_df_filt.to_csv('./id_seqs_text_2000_2022_filt.zip', index=False, compression='zip')
files.download('./id_seqs_text_2000_2022_filt.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
lat_long_df_filt = lat_long_df[lat_long_df["ID"].isin(result_df_filt["ID"].values)]

,ID,latitude,longitude
1755,1755,-20.001389,-70.123611
1756,1756,-20.001389,-70.120834
1757,1757,-20.001389,-70.118057
1758,1758,-20.001389,-70.115280
1759,1759,-20.001389,-70.112503


In [ ]:
lat_long_df_filt.to_csv('./lat_long_df_filt.zip', index=False, compression='zip')
files.download('./lat_long_df_filt.zip')

### Interactive Map for Bounding Box Visualization and Polygon Drawing

This map displays the original bounding box (`minx, miny, maxx, maxy`) and provides drawing tools to select new regions.

**Instructions:**
1.  **Draw polygons** using the drawing tools on the left side of the map.
2.  Once you're done drawing, click the **'Save GeoJSON'** icon (looks like a floppy disk) in the draw toolbar.
3.  This will download a GeoJSON file or display its content. **Copy the content of the GeoJSON.**
4.  Paste the copied GeoJSON string into the `drawn_polygons_geojson` variable in the next code cell. Ensure it's enclosed within triple quotes (`'''...'''`).
5.  Run the subsequent cells to process your drawn polygons and crop the dataset.

In [ ]:
import folium
from folium.plugins import Draw

# Calculate the center of the bounding box for map centering
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

# Create a Folium map centered on the bounding box with an appropriate zoom
m = folium.Map(location=[center_lat, center_lon], zoom_start=8)

# Add the original bounding box as a rectangle for reference
folium.Rectangle(
    bounds=[[miny, minx], [maxy, maxx]],
    color="#ff7800",
    fill=True,
    fill_color="#ffff00",
    fill_opacity=0.2,
    tooltip="Original Bounding Box"
).add_to(m)

# Add the Draw control to the map
Draw(export=True).add_to(m)

# Display the map
m

### Paste Exported GeoJSON Here

After drawing your polygons on the map above and exporting the GeoJSON, paste its content into the variable below. Remember to replace the example GeoJSON with your actual data.

In [ ]:
import json
from google.colab import files

print("Please upload your GeoJSON file (e.g., drawn_shapes.geojson).")
uploaded = files.upload()

# Assuming only one file is uploaded
if uploaded:
    filename = next(iter(uploaded))
    drawn_polygons_geojson = uploaded[filename].decode('utf-8')
    print(f"File '{filename}' loaded successfully.")
else:
    drawn_polygons_geojson = '''{}''' # Fallback to empty JSON if no file is uploaded
    print("No file uploaded. Using empty GeoJSON.")

Please upload your GeoJSON file (e.g., drawn_shapes.geojson).


Saving data.geojson to data.geojson
File 'data.geojson' loaded successfully.


### Extract Bounding Boxes and Crop `ds`

This cell will parse the GeoJSON you've provided, extract the bounding box coordinates for each polygon, and then use these to crop the `ds` xarray dataset. If multiple polygons are provided, the `ds` dataset will be cropped to the union of their bounding boxes for simplicity, or you can modify the code to crop to each individually.

In [ ]:
import json
from shapely.geometry import shape

def get_union_bbox_from_geojson(geojson_string):
    """
    Parses a GeoJSON string, extracts all polygon geometries, and calculates
    the union of their bounding boxes.

    Args:
        geojson_string: A string containing GeoJSON data (FeatureCollection).

    Returns:
        A tuple (minx, miny, maxx, maxy) representing the union bounding box,
        or None if no valid polygons are found.
    """
    try:
        geojson_data = json.loads(geojson_string)
        min_lon, min_lat, max_lon, max_lat = float('inf'), float('inf'), float('-inf'), float('-inf')
        found_polygon = False

        if geojson_data and 'features' in geojson_data:
            for feature in geojson_data['features']:
                if 'geometry' in feature and feature['geometry']['type'] == 'Polygon':
                    polygon = shape(feature['geometry'])
                    if not polygon.is_empty:
                        bounds = polygon.bounds
                        min_lon = min(min_lon, bounds[0])
                        min_lat = min(min_lat, bounds[1])
                        max_lon = max(max_lon, bounds[2])
                        max_lat = max(max_lat, bounds[3])
                        found_polygon = True

        if found_polygon:
            return (min_lon, min_lat, max_lon, max_lat)
        else:
            print("No valid Polygon features found in the GeoJSON.")
            return None

    except json.JSONDecodeError:
        print("Error: Invalid GeoJSON string. Please check your pasted content.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

# Get the union bounding box from the drawn polygons
union_bbox = get_union_bbox_from_geojson(drawn_polygons_geojson)

if union_bbox:
    bbox_minx, bbox_miny, bbox_maxx, bbox_maxy = union_bbox

    print(f"Union Bounding Box Coordinates: minx={bbox_minx}, miny={bbox_miny}, maxx={bbox_maxx}, maxy={bbox_maxy}")

    # Crop the dataset using the union bounding box
    # Note: xarray's sel for lat often expects max_lat to min_lat for 'decreasing' order
    ds_cropped_by_drawn_polygons = ds.sel(lon=slice(bbox_minx, bbox_maxx), lat=slice(bbox_maxy, bbox_miny))

    print("\nDataset cropped successfully. Displaying info of the new cropped dataset:")
    display(ds_cropped_by_drawn_polygons)
else:
    print("Could not extract bounding box. Please ensure you have pasted valid GeoJSON containing polygons.")

Union Bounding Box Coordinates: minx=-63.583374, miny=-29.623609, maxx=-59.067993, maxy=-25.372568

Dataset cropped successfully. Displaying info of the new cropped dataset:


<xarray.Dataset> Size: 229MB
Dimensions:     (time: 23, lat: 1530, lon: 1626)
Coordinates:
  * time        (time) datetime64[ns] 184B 2000-01-01 2001-01-01 ... 2022-01-01
  * lat         (lat) float32 6kB -25.37 -25.38 -25.38 ... -29.62 -29.62 -29.62
  * lon         (lon) float32 7kB -63.58 -63.58 -63.58 ... -59.07 -59.07 -59.07
Data variables:
    lccs_class  (time, lat, lon) float32 229MB ...
Attributes: (12/38)
    id:                         ESACCI-LC-L4-LCCS-Map-300m-P1Y-2000-v2.0.7cds
    title:                      Land Cover Map of ESA CCI brokered by CDS
    summary:                    This dataset characterizes the land cover of ...
    type:                       ESACCI-LC-L4-LCCS-Map-300m-P1Y
    project:                    Climate Change Initiative - European Space Ag...
    references:                 http://www.esa-landcover-cci.org/
    ...                         ...
    geospatial_lon_max:         180
    spatial_resolution:         300m
    geospatial_lat_units:       degrees_north
    geospatial_lat_resolution:  0.002778
    geospatial_lon_units:       degrees_east
    geospatial_lon_resolution:  0.002778

In [ ]:
ds_cropped_by_drawn_polygons

<xarray.Dataset> Size: 229MB
Dimensions:     (time: 23, lat: 1530, lon: 1626)
Coordinates:
  * time        (time) datetime64[ns] 184B 2000-01-01 2001-01-01 ... 2022-01-01
  * lat         (lat) float32 6kB -25.37 -25.38 -25.38 ... -29.62 -29.62 -29.62
  * lon         (lon) float32 7kB -63.58 -63.58 -63.58 ... -59.07 -59.07 -59.07
Data variables:
    lccs_class  (time, lat, lon) float32 229MB ...
Attributes: (12/38)
    id:                         ESACCI-LC-L4-LCCS-Map-300m-P1Y-2000-v2.0.7cds
    title:                      Land Cover Map of ESA CCI brokered by CDS
    summary:                    This dataset characterizes the land cover of ...
    type:                       ESACCI-LC-L4-LCCS-Map-300m-P1Y
    project:                    Climate Change Initiative - European Space Ag...
    references:                 http://www.esa-landcover-cci.org/
    ...                         ...
    geospatial_lon_max:         180
    spatial_resolution:         300m
    geospatial_lat_units:       degrees_north
    geospatial_lat_resolution:  0.002778
    geospatial_lon_units:       degrees_east
    geospatial_lon_resolution:  0.002778

In [ ]:
lat_long_ds_cropped_by_drawn_polygons = create_lat_long_df(ds_cropped_by_drawn_polygons)

In [ ]:
lat_long_ds_cropped_by_drawn_polygons

,ID,latitude,longitude
0,0,-25.373611,-63.581944
1,1,-25.373611,-63.579166
2,2,-25.373611,-63.576389
3,3,-25.373611,-63.573612
4,4,-25.373611,-63.570835
...,...,...,...
2487775,2487775,-29.620832,-59.079166
2487776,2487776,-29.620832,-59.076389
2487777,2487777,-29.620832,-59.073612
2487778,2487778,-29.620832,-59.070835


In [ ]:
lccs_class_ds_cropped_by_drawn_polygons = create_lccs_class_df(ds_cropped_by_drawn_polygons)

In [ ]:
lccs_class_ds_cropped_by_drawn_polygons

,lccs_class_2000,lccs_class_2001,lccs_class_2002,lccs_class_2003,lccs_class_2004,lccs_class_2005,lccs_class_2006,lccs_class_2007,lccs_class_2008,lccs_class_2009,...,lccs_class_2014,lccs_class_2015,lccs_class_2016,lccs_class_2017,lccs_class_2018,lccs_class_2019,lccs_class_2020,lccs_class_2021,lccs_class_2022,ID
0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,0
1,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1
2,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2
3,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,3
4,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2487775,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2487775
2487776,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2487776
2487777,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,...,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,2487777
2487778,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,...,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,2487778


In [ ]:
%%time

# Mapping of numeric codes to letters (using integers as keys)
code_mapping = {
    1: 'A',
    2: 'F',
    3: 'G',
    4: 'Wt',
    5: 'U',
    6: 'Sh',
    7: 'Sp',
    8: 'B',
    9: 'Wa',
    0: 'Nd'
}

# Get all lccs_class columns
lccs_cols = [col for col in lccs_class_ds_cropped_by_drawn_polygons.columns if 'lccs_class' in col]

# Process in chunks to avoid memory overflow
chunk_size = 100000  # Adjust if still crashing (try 50000)
result_list = []

for start_idx in range(0, len(lccs_class_ds_cropped_by_drawn_polygons), chunk_size):
    end_idx = min(start_idx + chunk_size, len(lccs_class_ds_cropped_by_drawn_polygons))
    chunk = lccs_class_ds_cropped_by_drawn_polygons.iloc[start_idx:end_idx]

    # Apply mapping directly to integer values, then convert to string
    arrays = []
    for col in lccs_cols:
        # Map the integer values directly using the dict
        mapped_col = chunk[col].map(code_mapping).astype(str)
        arrays.append(mapped_col.values)

    # Concatenate with '-' using a simpler approach
    seqs = []
    for i in range(len(chunk)):
        seq = '-'.join([str(arrays[j][i]) for j in range(len(arrays))])
        seqs.append(seq)

    # Create chunk result
    chunk_result = pd.DataFrame({
        'ID': chunk['ID'].values,
        'seqs': seqs
    })

    result_list.append(chunk_result)

    print(f"Processed {end_idx} rows...")

Processed 100000 rows...
Processed 200000 rows...
Processed 300000 rows...
Processed 400000 rows...
Processed 500000 rows...
Processed 600000 rows...
Processed 700000 rows...
Processed 800000 rows...
Processed 900000 rows...
Processed 1000000 rows...
Processed 1100000 rows...
Processed 1200000 rows...
Processed 1300000 rows...
Processed 1400000 rows...
Processed 1500000 rows...
Processed 1600000 rows...
Processed 1700000 rows...
Processed 1800000 rows...
Processed 1900000 rows...
Processed 2000000 rows...
Processed 2100000 rows...
Processed 2200000 rows...
Processed 2300000 rows...
Processed 2400000 rows...
Processed 2487780 rows...
CPU times: user 12.1 s, sys: 114 ms, total: 12.2 s
Wall time: 12.2 s


In [ ]:
# Concatenate all chunks
result_df_lccs_class_ds_cropped_by_drawn_polygons = pd.concat(result_list, ignore_index=True)

print(result_df_lccs_class_ds_cropped_by_drawn_polygons.head())
print(f"Total rows: {len(result_df_lccs_class_ds_cropped_by_drawn_polygons)}")

   ID                                           seqs
0   0  F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F
1   1  F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F
2   2  F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F
3   3  F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F
4   4  F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F
Total rows: 2487780


In [ ]:
result_df_lccs_class_ds_cropped_by_drawn_polygons

,ID,seqs
0,0,F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F
1,1,F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F
2,2,F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F
3,3,F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F
4,4,F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F-F
...,...,...
2487775,2487775,A-A-A-A-A-A-A-A-A-A-A-A-A-A-A-A-A-A-A-A-A-A-A
2487776,2487776,A-A-A-A-A-A-A-A-A-A-F-F-F-F-F-F-F-F-F-F-F-F-F
2487777,2487777,Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-W...
2487778,2487778,Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-Wa-W...


In [ ]:
lat_long_ds_cropped_by_drawn_polygons.to_csv('./lat_long_df_test_set.zip', index=False, compression='zip')
files.download('./lat_long_df_test_set.zip')

result_df_lccs_class_ds_cropped_by_drawn_polygons.to_csv('./id_seqs_text_2000_2022_test_set.zip', index=False, compression='zip')
files.download('./id_seqs_text_2000_2022_test_set.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>